In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,r2_score

In [5]:
data = pd.read_csv('student_chart.csv')

In [6]:
X = data.select_dtypes(include=np.number)
X = X.drop(columns=['exam_score','student_id'])
y = data['exam_score']

In [7]:
from IPython.display import Markdown
def performance(X,message):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    display(Markdown(f'***{message}***'))
    display(Markdown(f"**MSE :** {mean_squared_error(y_test,y_pred)}"))
    display(Markdown(f"**R2 Score :** {r2_score(y_test,y_pred)}"))

#Before Feature Selection

In [8]:
model = LinearRegression()
performance(X,message='Before Feature Selection')

***Before Feature Selection***

**MSE :** 25.013716673267062

**R2 Score :** 0.8164265753752491

#Correlation Method

In [9]:
correlation = X.corrwith(y)
print(correlation.sort_values(ascending=False))
# Get column names where any correlation > 0.5
X_corr = correlation[correlation > 0.5].index

print('Correlation Features :',X_corr)
performance(X[X_corr],message='After Correlation')

productivity_score              0.886401
focus_index                     0.749879
mental_health_score             0.546547
study_hours                     0.513434
total_study_hours               0.436046
sleep_hours                     0.234710
self_study_hours                0.083602
exercise_minutes                0.040724
Gender_label                    0.018703
internet_label                  0.014452
online_classes_hours            0.004975
education_label                -0.005366
age                            -0.009047
gaming_hours                   -0.054506
caffeine_intake_mg             -0.075586
total_smart_device_usage_hrs   -0.101471
social_media_hours             -0.106116
screen_time_hours              -0.131940
part_time_job                  -0.149807
avg_screen_time                -0.168380
upcoming_deadline              -0.215342
burnout_level                  -0.407808
dtype: float64
Correlation Features : Index(['study_hours', 'mental_health_score', 'focus_index',


***After Correlation***

**MSE :** 28.31708361991125

**R2 Score :** 0.7921834614426535

#RFE Method

In [10]:
rfe = RFE(model, n_features_to_select=4)
X_wrapper = rfe.fit_transform(X, y)
selected_features_rfe = X.columns[rfe.support_]
print("\nWrapper Selected Features:",selected_features_rfe)
performance(X[selected_features_rfe],'Wrapper Method(RFE)')


Wrapper Selected Features: Index(['upcoming_deadline', 'focus_index', 'burnout_level',
       'productivity_score'],
      dtype='object')


***Wrapper Method(RFE)***

**MSE :** 24.768214398243305

**R2 Score :** 0.8182282945666816

#
#
#Embedded Method using RandomForestRegressor

In [15]:
random_forest_regression = RandomForestRegressor()
random_forest_regression.fit(X, y)

feature_importance = pd.Series(random_forest_regression.feature_importances_, 
                               index=X.columns)
top_features = feature_importance.sort_values(ascending=False).head(5)
print("Top Features :\n",top_features)
X_rfr = top_features.index
performance(X[X_rfr],'Embedded method ( RandomForestRegressor )')

Top Features :
 productivity_score    0.803506
burnout_level         0.048868
focus_index           0.029716
exercise_minutes      0.009732
sleep_hours           0.009587
dtype: float64


***Embedded method ( RandomForestRegressor )***

**MSE :** 24.831632238945296

**R2 Score :** 0.8177628767180651

#Standard Scalar

In [12]:
scaler_standard = StandardScaler()
X_train_standard = scaler_standard.fit_transform(X)
print(X_train_standard[:5])
performance(X_train_standard,'Scalar Standard')

[[-0.8781517   1.7021331  -0.77999477  0.19111052  0.03536853  0.56314776
  -0.42669503 -0.20498599  0.15058703 -1.48406371  1.0036065  -1.00280393
   1.56598907  1.36704605 -0.97193138  2.15947921  0.99491124  0.41538964
   0.1110753  -1.22930556 -1.22372503 -1.19963314]
 [-0.8781517  -1.27895477 -0.21966224  0.08946464 -0.91843865  0.88726884
  -0.89937589 -0.37393446  0.84943162  0.60870902 -0.99640646 -1.00280393
  -0.87400636 -1.35632842 -0.60478929 -1.39886745 -1.0497536  -0.13179713
  -0.37589444 -1.22930556 -1.22372503 -1.19963314]
 [ 0.51551867 -0.59819069 -2.10441711 -1.75032581 -1.12963881  0.46411077
   1.1804199   0.25761102 -0.15224562  0.10115949 -0.99640646  0.99720391
   0.86884752 -0.20494231 -0.78941332  0.46785486 -2.22113244 -1.37496709
  -0.69429772  0.00518486 -1.22372503 -1.19963314]
 [-1.22656929  0.66451688 -0.33852065  1.01444216 -0.49603833  0.57215113
  -0.60717318  1.88675701  0.89602126  1.58904442  1.0036065   0.99720391
  -0.87400636 -0.7148849   2.2249

***Scalar Standard***

**MSE :** 25.013716673267066

**R2 Score :** 0.8164265753752491

#MinMax Method

In [13]:
# Min-Max Scaling
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X)

print(X_train_minmax[:5])
performance(X_train_minmax,'MinMax Method')

[[0.22222222 0.64527027 0.21052632 0.36666667 0.36835749 0.38829787
  0.42       0.38251748 0.54362416 0.0761523  1.         0.
  1.         0.67301536 0.31859598 0.74881468 0.66452854 0.53357664
  0.47037375 0.         0.         0.        ]
 [0.22222222 0.18665541 0.29959514 0.35       0.19927536 0.45212766
  0.32833333 0.35314685 0.74496644 0.67935872 0.         0.
  0.22222222 0.23879641 0.37274798 0.13090085 0.35214881 0.45036496
  0.39927074 0.         0.         0.        ]
 [0.66666667 0.29138514 0.         0.04833333 0.16183575 0.36879433
  0.73166667 0.46293706 0.45637584 0.53306613 0.         1.
  0.77777778 0.42237516 0.34551667 0.45506081 0.17318794 0.26131387
  0.35278031 0.5        0.         0.        ]
 [0.11111111 0.48564189 0.28070175 0.50166667 0.27415459 0.39007092
  0.385      0.74615385 0.75838926 0.96192385 1.         1.
  0.22222222 0.34106914 0.79012218 0.20531849 0.62860808 0.53649635
  0.70920693 0.         0.         0.5       ]
 [0.33333333 0.57685811 0.23

***MinMax Method***

**MSE :** 25.013716673267076

**R2 Score :** 0.8164265753752491

###Feature Scaling Completed Successfully